In [11]:
from prefect import flow, task
import psycopg2
import os
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# DB CONNECTION
# -------------------------------
def get_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT"),
        sslmode="require"
    )

def run_query(query):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute(query)
    conn.commit()
    cur.close()
    conn.close()

In [13]:
# CREATE CLEAN TABLE
# -------------------------------
@task
def create_weather_clean():
    query = """
    DROP TABLE IF EXISTS weather_clean;

    CREATE TABLE weather_clean AS
    SELECT
        l.city_name,
        w.location_id,
        w.timestamp,
        w.temperature,
        w.humidity,
        w.wind_speed,
        w.precipitation::DOUBLE PRECISION AS precipitation,
        w.wind_direction,
        w.cloud_cover,
        w.dew_point,
        w.apparent_temperature,
        w.pressure_msl,
        w.weather_code
    FROM weather_data w
    JOIN locations l ON w.location_id = l.location_id
    WHERE w.temperature IS NOT NULL;
    """
    run_query(query)
    print("weather_clean created")

In [14]:
# FEATURE ENGINEERING
# -------------------------------
@task
def create_weather_features():
    query = """
    DROP VIEW IF EXISTS city_rankings;
    DROP TABLE IF EXISTS weather_features;

    CREATE TABLE weather_features AS
    SELECT
        city_name,
        location_id,
        timestamp,
        temperature,
        humidity,
        wind_speed,
        precipitation,
        EXTRACT(HOUR FROM timestamp) AS hour,
        EXTRACT(DAY FROM timestamp) AS day,
        EXTRACT(MONTH FROM timestamp) AS month,
        CASE
            WHEN EXTRACT(MONTH FROM timestamp) IN (12,1,2) THEN 'Winter'
            WHEN EXTRACT(MONTH FROM timestamp) IN (3,4,5) THEN 'Spring'
            WHEN EXTRACT(MONTH FROM timestamp) IN (6,7,8) THEN 'Summer'
            ELSE 'Autumn'
        END AS season
    FROM weather_clean;
    """
    run_query(query)
    print("weather_features created")

In [ ]:
# DAILY SUMMARY
# -------------------------------
@task
def create_daily_summary():
    query = """
    DROP TABLE IF EXISTS weather_daily_summary;

    CREATE TABLE weather_daily_summary AS
    SELECT
        city_name,
        DATE(timestamp) AS date,
        AVG(temperature) AS avg_temperature,
        AVG(humidity) AS avg_humidity,
        SUM(precipitation) AS total_precipitation,
        AVG(wind_speed) AS avg_wind_speed
    FROM weather_features
    GROUP BY city_name, DATE(timestamp);
    """
    run_query(query)
    print("weather_daily_summary created")

In [16]:
# CITY RANKINGS
# -------------------------------
@task
def create_city_rankings():
    query = """
    DROP VIEW IF EXISTS city_rankings;

    CREATE VIEW city_rankings AS
    SELECT
        city_name,
        avg_temp,
        RANK() OVER (ORDER BY avg_temp DESC) AS temp_rank
    FROM (
        SELECT
            city_name,
            AVG(temperature) AS avg_temp
        FROM weather_features
        GROUP BY city_name
    ) t;
    """
    run_query(query)
    print("city_rankings view created")

In [17]:
# ANOMALY DETECTION
# -------------------------------
@task
def create_anomalies():
    query = """
    DROP TABLE IF EXISTS weather_anomalies;

    CREATE TABLE weather_anomalies AS
    SELECT *
    FROM (
        SELECT
            city_name,
            location_id,
            timestamp,
            temperature,
            AVG(temperature) OVER (PARTITION BY location_id) AS avg_temp,
            temperature - AVG(temperature) OVER (PARTITION BY location_id) AS deviation
        FROM weather_features
    ) sub
    WHERE ABS(deviation) > 5;
    """
    run_query(query)
    print("weather_anomalies created")

In [18]:
# MAIN FLOW
# -------------------------------
@flow(name="weather-etl-pipeline")
def weather_pipeline():
    clean = create_weather_clean()
    features = create_weather_features(wait_for=[clean])
    summary = create_daily_summary(wait_for=[features])
    rankings = create_city_rankings(wait_for=[features])
    anomalies = create_anomalies(wait_for=[features])

In [19]:
# RUN
# -------------------------------
if __name__ == "__main__":
    weather_pipeline()

23:29:53.917 | INFO    | Flow run 'grinning-gopher' - Beginning flow run 'grinning-gopher' for flow 'weather-etl-pipeline'

weather_clean created


23:30:02.968 | INFO    | Task run 'create_weather_clean-2e4' - Finished in state Completed()

weather_features created


23:30:07.806 | INFO    | Task run 'create_weather_features-6ef' - Finished in state Completed()

...

city_rankings view created


23:30:13.712 | INFO    | Task run 'create_city_rankings-7f7' - Finished in state Completed()

weather_anomalies created


23:30:17.624 | INFO    | Task run 'create_anomalies-324' - Finished in state Completed()

23:30:17.944 | INFO    | Flow run 'grinning-gopher' - Finished in state Completed()